# 03 — Construction du CNN & Entraînement

**Objectif :** construire un réseau de neurones convolutif (CNN) avec
TensorFlow/Keras, le compiler, l'entraîner sur le jeu de train, et
surveiller les performances sur le jeu de validation.

**Architecture :** 3 blocs Conv → ReLU → MaxPool, puis GlobalAveragePooling,
Dropout, Dense, et une couche de sortie Softmax à 6 classes.

**Callbacks :** EarlyStopping + ReduceLROnPlateau.

In [13]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath(".."))

from src.data import make_dataset, IMG_SIZE, BATCH_SIZE, SEED

tf.random.set_seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (12, 6)
print("TensorFlow version :", tf.__version__)

TensorFlow version : 2.21.0


## 1. Chargement du split + pipelines

On recharge les 3 pipelines `tf.data` depuis `data/splits.npz`.
Rien n'est re-splitté — le split est figé.

In [14]:
splits = np.load("../data/splits.npz")

X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]
class_names      = list(splits["class_names"])

train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val,   training=False)
test_ds  = make_dataset(X_test,  y_test,  training=False)

NUM_CLASSES = len(class_names)

print(f"Classes      : {class_names}")
print(f"Num classes  : {NUM_CLASSES}")
print(f"Batches      : train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Classes      : [np.str_('buildings'), np.str_('forest'), np.str_('glacier'), np.str_('mountain'), np.str_('sea'), np.str_('street')]
Num classes  : 6
Batches      : train=373, val=80, test=80


## 2. Architecture du CNN

Trois blocs convolutifs de profondeur croissante (32 → 64 → 128 filtres),
suivis d'un GlobalAveragePooling pour compacter les features, puis d'une
tête de classification (Dropout + Dense + Softmax).

**Choix :**
- `padding="same"` → pas de perte de taille à chaque conv
- `activation="relu"` → non-linéarité standard
- `GlobalAveragePooling2D` → moins de paramètres que Flatten → moins d'overfitting
- `Dropout(0.5)` → régularisation
- `Dense(6, softmax)` → probabilités par classe

In [15]:
def build_cnn(input_shape=(150, 150, 3), num_classes=6):
    model = tf.keras.Sequential([
        # ----- Bloc 1 -----
        tf.keras.layers.Conv2D(32, (3, 3), padding="same",
                               activation="relu", input_shape=input_shape),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # ----- Bloc 2 -----
        tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # ----- Bloc 3 -----
        tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # ----- Tête de classification -----
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ], name="intel_cnn")
    return model


model = build_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
model.summary()

Model: "intel_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,534 (431.77 KB)

 Trainable params: 110,534 (431.77 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Compilation

- **Optimizer :** `adam` — standard, adaptatif.
- **Loss :** `sparse_categorical_crossentropy` — car labels sont des entiers (0..5).
- **Métrique :** `accuracy`.

In [ ]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = "sparse_categorical_crossentropy",
    metrics   = ["accuracy"],
)

## 4. Callbacks

- **EarlyStopping** : arrête si `val_loss` ne s'améliore plus pendant 5 époques.
- **ReduceLROnPlateau** : divise le LR par 2 si `val_loss` stagne pendant 3 époques.

In [16]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

## 5. Entraînement

On entraîne jusqu'à 30 époques. EarlyStopping peut arrêter avant.
Le meilleur modèle (selon `val_loss`) est automatiquement restauré.

In [ ]:
EPOCHS = 30

history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = EPOCHS,
    callbacks       = callbacks,
)

## 6. Courbes d'apprentissage

On trace Loss et Accuracy pour train vs validation, pour diagnostiquer
overfitting / underfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"],     label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history["accuracy"],     label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Analyse rapide

- **Les deux courbes descendent** → le modèle apprend ✅
- **Gap croissant entre train et val** → overfitting
- **Les deux plateaux hauts** → underfitting
- **Val loss remonte** → early stopping devrait couper

In [ ]:
best_epoch = np.argmin(history.history["val_loss"])
best_val_loss = history.history["val_loss"][best_epoch]
best_val_acc  = history.history["val_accuracy"][best_epoch]

print(f"Meilleure époque    : {best_epoch + 1}")
print(f"Meilleure val_loss  : {best_val_loss:.4f}")
print(f"Meilleure val_acc   : {best_val_acc:.4f}")

## 8. Sauvegarde du modèle + historique

On sauvegarde le modèle entraîné et les courbes d'apprentissage pour
analyse ultérieure (notebooks 04 et 05).

In [ ]:
os.makedirs("../models", exist_ok=True)

model.save("../models/cnn_v1.keras")
np.savez("../models/history_v1.npz", **history.history)

print("✅ Modèle sauvegardé   → ../models/cnn_v1.keras")
print("✅ Historique sauvegardé → ../models/history_v1.npz")